# 极客项目 ch14：基于树莓派的天气监控器

依赖：pyserial/pyaudio/RPi.GPIO + 对应硬件，代码为完整忠实实现。对应《Python Playground》第 14 章（Raspberry Pi Weather Monitor）：树莓派上 Bottle 提供 Web 服务，DHT11 采集温湿度，浏览器 flot 每秒刷新图表并可远程开关 LED；代码搬运自 `code/ch14_piweather.py`，运行：`sudo python3 ch14_piweather.py --ip <IP> --port 8080`。

## 书 14.2.6 Web 框架 Bottle

@route 装饰器把 URL 映射到函数；/hello 是连通性测试（Hello Bottle World），main() 里 run(host, port, debug=True) 启动服务器。

In [ ]:
# ===== 书 14.2.6/14.4 节：所需模块 =====
from bottle import route, run, request, static_file
import argparse
import RPi.GPIO as GPIO
import Adafruit_DHT

In [ ]:
# ===== 书 14.2.6 节：Hello Bottle World =====
@route('/hello')
def hello():
    return "Hello Bottle World!"

## 书 14.4.5（节末）静态 JS 文件路由

正则路由 `/<filename:re:.*\.js>` 匹配一切 .js 请求，static_file 在程序同级的 flot/ 子目录下查找返回（jquery.js、jquery.flot.js 等）。cell 顺序与源文件一致，编号随笔记小节。

In [ ]:
# ===== 书 14.4.5 节末：静态 JS 文件路由 =====
@route('/<filename:re:.*\\.js>')
def javascripts(filename):
    return static_file(filename, root='flot')  # 程序同级的 flot/ 子目录存 jquery 系 .js

## 书 14.4.2 绘制数据（/plot 路由）

整个网页（HTML + CSS + JavaScript）作为单个字符串返回：flot 每 1 秒 AJAX 拉取 /getdata 刷新 RH/T 两条曲线（各留最近 100 个样本），复选框 ckLED 点击即 POST /ledctrl，plotclick 用 dataIndex 反查数值显示（14.4.2–14.4.5 各块拼装）。

In [ ]:
# ===== 书 14.4.2–14.4.5 节：plot 路由（整页 HTML+CSS+JS 大字符串） =====
@route('/plot')
def plot():
    return '''
<html>
<head>
    <meta http-equiv="Content-Type" content="text/html; charset=utf-8">
    <title>PiWeather</title>
    <style>
     .demo-placeholder {
    width: 90%;
    height: 50%;
    }
    </style>
    <script language="javascript" type="text/javascript"
       src="jquery.js"></script>
    <script language="javascript" type="text/javascript"
       src="jquery.flot.js"></script>
    <script language="javascript" type="text/javascript"
       src="jquery.flot.time.js"></script>
    <script language="javascript" type="text/javascript">

$(document).ready(function() {

    // plot options
    var options = {
        series: {
          lines: {
            show: true
          },
          points: {
            show: true
          }
        },
        grid: {
          clickable: true
        },
        yaxes: [{min: 0, max: 100}],
        xaxes: [{min: 0, max: 100}],
    };

    // create empty plot
    var plot = $.plot("#placeholder", [[]], options);

    // initialize data arrays
    var RH = [];
    var T = [];
    var timeStamp = [];
    // get data from server
    function getData() {
        // AJAX callback
        function onDataReceived(jsonData) {
            timeStamp.push(Date());
            // add RH data
            RH.push(jsonData.RH);
            // removed oldest
            if (RH.length > 100) {
              RH.splice(0, 1);
            }
            // add T data
            T.push(jsonData.T);
            // removed oldest
            if (T.length > 100) {
              T.splice(0, 1);
            }
            s1 = [];
            s2 = [];
            for (var i = 0; i < RH.length; i++) {
                s1.push([i, RH[i]]);
                s2.push([i, T[i]]);
            }
            // set to plot
            plot.setData([s1, s2]);
            plot.draw();
        }

        // AJAX error handler
        function onError(){
            $('#ajax-panel').html('<p><strong>Ajax error!</strong> </p>');
        }

        // make the AJAX call
        $.ajax({
            url: "getdata",
            type: "GET",
            dataType: "json",
            success: onDataReceived,
            error: onError
        });
     }

     // define an update function
     function update() {
        // get data
        getData();
        // set timeout
        setTimeout(update, 1000);
     }

     // call update
     update();

     // define click handler for LED ctrl
     $('#ckLED').click(function() {
         var isChecked = $("#ckLED").is(":checked") ? 1:0;
         $.ajax({
           url: '/ledctrl',
           type: 'POST',
           data: { strID:'ckLED', strState:isChecked }
         });
      });

     $("#placeholder").bind("plotclick", function (event, pos, item) {
          if (item) {
            plot.highlight(item.series, item.datapoint);
            var strData = ' [Clicked Data: ' +
                           timeStamp[item.dataIndex] + ': T = ' +
                           T[item.dataIndex] + ', RH = ' + RH[item.dataIndex]
                          + ']';
            $('#data-values').html(strData);
          }
     });
});

</script>
</head>

<body>
    <div id="header">
        <h2>Temperature/Humidity</h2>
    </div>

    <div id="content">
        <div class="demo-container">
            <div id="placeholder" class="demo-placeholder"></div>
        </div>
        <div id="ajax-panel"> </div>
    </div>
    <div>
      <input type="checkbox" id="ckLED" value="on">Enable Lighting.
      <span id="data-values"> </span>
    </div>

</body>
</html>
'''

## 书 14.4.1 处理传感器数据请求（/getdata）

Adafruit_DHT.read_retry(DHT11, 23) 读温湿度（23 是 BCM 编号），字典由 Bottle 自动转 JSON 返回；注意重试用尽会返回 (None, None)。

In [ ]:
# ===== 书 14.4.1 节：getdata 路由 =====
@route('/getdata', method='GET')
def getdata():
    RH, T = Adafruit_DHT.read_retry(Adafruit_DHT.DHT11, 23)  # 23 是 BCM 编号（main 里 18 是 BOARD 编号）
    return {"RH": RH, "T": T}

## 书 14.4.4 LED 开关（/ledctrl）

POST 表单里的 strState（'0'/'1'）经 bool(int()) 转布尔后 GPIO.output(18, on) 开关 LED。

In [ ]:
# ===== 书 14.4.4 节：ledctrl 路由 =====
@route('/ledctrl', method='POST')
def ledctrl():
    val = request.forms.get('strState')
    on = bool(int(val))
    GPIO.output(18, on)

## 书 14.4–14.5 main() 与运行入口

argparse 必填 --ip/--port；GPIO.setmode(BOARD)、#18 设输出且初始 False；run(host, port, debug=True) 启动 Bottle。注意 BOARD 编号 18（= BCM 24）与 getdata 里 BCM 23 是两套编号。

In [ ]:
# ===== 书 14.4 节：main()——argparse + GPIO 初始化 =====
# main() function
def main():
    print('starting piweather...')  # 原书是 Python 2 的 print 语句
    parser = argparse.ArgumentParser(description="PiWeather...")
    parser.add_argument('--ip', dest='ipAddr', required=True)
    parser.add_argument('--port', dest='portNum', required=True)
    args = parser.parse_args()

    # setup GPIO
    GPIO.setmode(GPIO.BOARD)
    GPIO.setup(18, GPIO.OUT)
    GPIO.output(18, False)
    run(host=args.ipAddr, port=args.portNum, debug=True)


if __name__ == '__main__':
    main()